In [1]:
import torch
import numpy as np
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

In [2]:
#Load Model 

MODEL_PATH = "sentiment_distilbert_best"

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_PATH)
model = DistilBertForSequenceClassification.from_pretrained(MODEL_PATH)

model.eval()

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): MultiHeadSelfAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
 

In [3]:
#Gradient & Explaination function

def explain_prediction(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]

    # Get embeddings
    embeddings = model.distilbert.embeddings.word_embeddings(input_ids)
    embeddings.retain_grad()
    embeddings.requires_grad_(True)

    # Forward pass using embeddings
    outputs = model(
        inputs_embeds=embeddings,
        attention_mask=attention_mask
    )

    pred_class = torch.argmax(outputs.logits, dim=1).item()
    score = outputs.logits[0, pred_class]

    # Backward pass
    model.zero_grad()
    score.backward()

    # Gradient × Input
    token_importance = (embeddings.grad * embeddings).sum(dim=-1).squeeze(0)

    tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze(0))
    scores = token_importance.detach().numpy()

    return tokens, scores, pred_class

In [4]:
def show_explanation(tokens, scores, threshold=0.02):
    for tok, score in zip(tokens, scores):
        if tok in ["[CLS]", "[SEP]", "[PAD]"]:
            continue
        if abs(score) > threshold:
            print(f"{tok:>12} : {score:+.3f}")

In [5]:
text = "The product is okay, not great but not terrible either."

tokens, scores, pred = explain_prediction(text)

label_map = {0: "Negative", 1: "Neutral", 2: "Positive"}
print("Predicted Sentiment:", label_map[pred])
print("\nImportant words:\n")

show_explanation(tokens, scores)

Predicted Sentiment: Neutral

Important words:

         the : -0.090
     product : +0.063
          is : -0.314
        okay : -0.182
           , : +0.045
         not : +0.043
       great : -0.027
         but : +0.466
         not : -0.916
    terrible : -0.707
      either : -0.108
           . : +0.280
